In [ ]:
!pip install -q unsloth

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-Coder-3B-Instruct",
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
)

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    ""HF DATASET HERE",
    split="train"
)

dataset = dataset.shuffle(seed=42)

In [ ]:
EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    texts = []

    for instruction, inp, output in zip(
        examples["instruction"],
        examples["input"],
        examples["output"]
    ):
        text = f"""### Instruction:
{instruction}

### Input:
{inp}

### Response:
{output}""" + EOS_TOKEN

        texts.append(text)

    return {"text": texts}

dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
)

In [ ]:
train_5k = dataset.select(range(5000))

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_5k,
    dataset_text_field="text",
    max_seq_length=max_seq_length,

    args=TrainingArguments(
        output_dir="outputs",

        per_device_train_batch_size=1,

        gradient_accumulation_steps=4,

        learning_rate=2e-4,

        num_train_epochs=2,

        logging_steps=20,

        save_strategy="no",

        report_to="none",
    ),
)

In [ ]:
trainer.train()

In [ ]:
model.save_pretrained("checkpoint_5k")
tokenizer.save_pretrained("checkpoint_5k")

In [ ]:
train_next_5k = dataset.select(range(5000, 10000))

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_next_5k,
    dataset_text_field="text",
    max_seq_length=max_seq_length,

    args=TrainingArguments(
        output_dir="outputs",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        num_train_epochs=1,
        logging_steps=20,
        save_strategy="no",
        report_to="none",
    ),
)

trainer.train()

In [ ]:
model.save_pretrained("checkpoint_5k")
tokenizer.save_pretrained("checkpoint_5k")

In [ ]:
train_next_5k = dataset.select(range(10000, 15000))

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_next_5k,
    dataset_text_field="text",
    max_seq_length=max_seq_length,

    args=TrainingArguments(
        output_dir="outputs",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        num_train_epochs=2,
        logging_steps=20,
        save_strategy="no",
        report_to="none",
    ),
)

In [ ]:
trainer.train()

In [ ]:
model.save_pretrained("checkpoint_15k")
tokenizer.save_pretrained("checkpoint_15k")

In [ ]:
!ls checkpoint_15k

In [ ]:
FastLanguageModel.for_inference(model)

prompt = """
### Instruction:
Write a Python function to find the second largest element in a list.

### Input:

### Response:
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.7,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
model.save_pretrained_gguf(
    "gguf_model",
    tokenizer,
)

In [ ]:
!ls -lh gguf_model_gguf

In [ ]:
from google.colab import files

files.download(
    "gguf_model_gguf/qwen2.5-coder-3b-instruct.Q8_0.gguf"
)

In [ ]:
from google.colab import files

files.download("VCoder-LoRA.zip")